# Extraccion de accelerometria, PPG y ground truth PSG de DREAMT

Este notebook prepara datos de DREAMT para entrenar un modelo de clasificacion de fases de sueno usando senales wearable como entrada:

- `ACC_X`, `ACC_Y`, `ACC_Z`: accelerometria del Empatica E4.
- `BVP`: senal PPG/BVP del sensor de fotopletismografia.
- `Sleep_Stage`: etiqueta anotada por tecnico a partir de PSG, usada como ground truth.
- Canales PSG alineados, opcionales, para trazabilidad o modelos multimodales.

DREAMT es de acceso restringido en PhysioNet. Descarga el dataset tras aceptar el acuerdo de uso y coloca la carpeta local en `../data/dreamt` o cambia `DREAMT_ROOT` en la celda de configuracion.

Nota: `data_100Hz` contiene wearable + PSG alineados. `data_64Hz` contiene solo wearable y etiquetas, por lo que este notebook usa `data_100Hz` por defecto.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

## 1. Configuracion

Ajusta estas rutas a tu descarga local. Para entrenamiento con PSG como referencia, usa `data_100Hz`.

In [ ]:
DREAMT_ROOT = Path("../data/dreamt").resolve()
DATA_SUBDIR = "data_100Hz"
OUTPUT_DIR = Path("../data/processed/dreamt_wearable_psg").resolve()

EPOCH_SECONDS = 30
SAMPLE_RATE_HZ = 100
MIN_EPOCH_COVERAGE = 0.80

WEARABLE_COLUMNS = ["ACC_X", "ACC_Y", "ACC_Z", "BVP"]
LABEL_COLUMN = "Sleep_Stage"

LABEL_MAP_5_CLASS = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}
LABEL_MAP_3_CLASS = {"W": 0, "N1": 1, "N2": 1, "N3": 1, "R": 2}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DREAMT_ROOT, OUTPUT_DIR

## 2. Utilidades de descubrimiento y lectura

Las versiones de DREAMT pueden variar en nombres de archivo o compresion. Estas funciones buscan tablas `csv`, `parquet` o `feather`, normalizan nombres de columnas y detectan las senales esperadas.

In [ ]:
TABLE_SUFFIXES = {".csv", ".txt", ".tsv", ".parquet", ".feather", ".gz"}

def normalise_column_name(name: str) -> str:
    cleaned = str(name).strip()
    cleaned = re.sub(r"\s*\[[^\]]+\]", "", cleaned)
    cleaned = cleaned.replace("-", "_").replace(" ", "_")
    cleaned = re.sub(r"_+", "_", cleaned)
    return cleaned

def list_candidate_files(root: Path, data_subdir: str = DATA_SUBDIR) -> list[Path]:
    data_dir = root / data_subdir
    if not data_dir.exists():
        raise FileNotFoundError(
            f"No existe {data_dir}. Coloca ahi la descarga de DREAMT o cambia DREAMT_ROOT/DATA_SUBDIR."
        )
    files = []
    for path in data_dir.rglob("*"):
        suffixes = {s.lower() for s in path.suffixes}
        if path.is_file() and (suffixes & TABLE_SUFFIXES):
            files.append(path)
    return sorted(files)

def read_table(path: Path, nrows: int | None = None) -> pd.DataFrame:
    suffixes = [s.lower() for s in path.suffixes]
    if ".parquet" in suffixes:
        if nrows is not None:
            df = pd.read_parquet(path).head(nrows)
        else:
            df = pd.read_parquet(path)
    elif ".feather" in suffixes:
        if nrows is not None:
            df = pd.read_feather(path).head(nrows)
        else:
            df = pd.read_feather(path)
    else:
        sep = "\t" if path.suffix.lower() == ".tsv" else None
        df = pd.read_csv(path, sep=sep, engine="python", nrows=nrows)
    df = df.rename(columns={col: normalise_column_name(col) for col in df.columns})
    return df

def preview_files(files: list[Path], n: int = 10) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "file": [str(path.relative_to(DREAMT_ROOT)) for path in files[:n]],
            "size_mb": [round(path.stat().st_size / 1024**2, 2) for path in files[:n]],
        }
    )

candidate_files = list_candidate_files(DREAMT_ROOT)
print(f"Tablas encontradas en {DATA_SUBDIR}: {len(candidate_files)}")
preview_files(candidate_files, n=12)

In [ ]:
def find_signal_columns(df: pd.DataFrame) -> dict[str, str]:
    columns_upper = {col.upper(): col for col in df.columns}
    aliases = {
        "TIMESTAMP": ["TIMESTAMP", "TIME", "TIME_S", "SECONDS"],
        "ACC_X": ["ACC_X", "ACCX", "X"],
        "ACC_Y": ["ACC_Y", "ACCY", "Y"],
        "ACC_Z": ["ACC_Z", "ACCZ", "Z"],
        "BVP": ["BVP", "PPG", "BLOOD_VOLUME_PULSE"],
        "Sleep_Stage": ["SLEEP_STAGE", "SLEEPSTAGE", "STAGE", "LABEL"],
    }
    found = {}
    for canonical, options in aliases.items():
        for option in options:
            if option.upper() in columns_upper:
                found[canonical] = columns_upper[option.upper()]
                break
    return found

def is_dreamt_signal_table(path: Path) -> bool:
    try:
        df_head = read_table(path, nrows=5)
    except Exception as exc:
        warnings.warn(f"No se pudo leer {path.name}: {exc}")
        return False
    found = find_signal_columns(df_head)
    required = set(WEARABLE_COLUMNS + [LABEL_COLUMN])
    return required.issubset(found)

signal_files = [path for path in candidate_files if is_dreamt_signal_table(path)]
print(f"Tablas con wearable + etiqueta detectadas: {len(signal_files)}")
preview_files(signal_files, n=12)

## 3. Cargar un participante y separar wearable, PSG y etiqueta

El ground truth para clasificacion es `Sleep_Stage`, anotado desde PSG. Los canales PSG se conservan aparte para inspeccion, validacion de alineamiento o modelos multimodales, pero no deberian mezclarse como entrada si el objetivo es estimar sueno solo desde wearable.

In [ ]:
def infer_subject_id(path: Path) -> str:
    stem = path.name
    stem = re.sub(r"(\.csv|\.tsv|\.txt|\.parquet|\.feather|\.gz)+$", "", stem, flags=re.I)
    match = re.search(r"(S?\d{3,}|participant[_-]?\d+|subject[_-]?\d+)", stem, flags=re.I)
    return match.group(1) if match else stem

def load_record(path: Path) -> tuple[str, pd.DataFrame, dict[str, str]]:
    df = read_table(path)
    found = find_signal_columns(df)
    missing = [col for col in ["TIMESTAMP", *WEARABLE_COLUMNS, LABEL_COLUMN] if col not in found]
    if missing:
        raise ValueError(f"Faltan columnas {missing} en {path}")

    rename = {source: canonical for canonical, source in found.items()}
    df = df.rename(columns=rename)
    subject_id = infer_subject_id(path)
    df.insert(0, "subject_id", subject_id)

    df["TIMESTAMP"] = pd.to_numeric(df["TIMESTAMP"], errors="coerce")
    for col in WEARABLE_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype("string").str.strip()
    return subject_id, df, found

def split_wearable_psg_label(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    base = ["subject_id", "TIMESTAMP", *WEARABLE_COLUMNS, LABEL_COLUMN]
    psg_columns = [col for col in df.columns if col not in base]
    tidy = df[base + psg_columns].copy()
    tidy = tidy.dropna(subset=["TIMESTAMP"])
    return tidy, psg_columns

example_path = signal_files[0]
subject_id, example_df, found_columns = load_record(example_path)
example_df, psg_columns = split_wearable_psg_label(example_df)

print("Participante ejemplo:", subject_id)
print("Columnas PSG detectadas:", psg_columns[:20], "..." if len(psg_columns) > 20 else "")
display(example_df.head())
display(example_df[WEARABLE_COLUMNS + [LABEL_COLUMN]].describe(include="all"))

## 4. Crear ventanas de 30 segundos

Cada muestra queda asignada a un epoch PSG de 30 s. Para modelos secuenciales se exporta un tensor `X_wearable` con forma:

`n_epochs x samples_per_epoch x 4`

y un vector `y` con la etiqueta de PSG. Si hay canales PSG disponibles, se exporta tambien `X_psg` con la misma segmentacion temporal.

In [ ]:
def majority_label(values: pd.Series) -> str | None:
    clean = values.dropna().astype(str)
    clean = clean[~clean.str.lower().isin({"missing", "nan", "none", ""})]
    if clean.empty:
        return None
    return Counter(clean).most_common(1)[0][0]

def epochise_record(
    df: pd.DataFrame,
    psg_columns: list[str],
    sample_rate_hz: int = SAMPLE_RATE_HZ,
    epoch_seconds: int = EPOCH_SECONDS,
    min_coverage: float = MIN_EPOCH_COVERAGE,
    label_map: dict[str, int] = LABEL_MAP_5_CLASS,
) -> tuple[np.ndarray, np.ndarray, np.ndarray | None, pd.DataFrame]:
    samples_per_epoch = sample_rate_hz * epoch_seconds
    work = df.sort_values("TIMESTAMP").copy()
    work["epoch_index"] = np.floor(work["TIMESTAMP"] / epoch_seconds).astype("int64")

    x_wearable, y, x_psg, meta_rows = [], [], [], []
    numeric_psg_columns = [col for col in psg_columns if pd.api.types.is_numeric_dtype(work[col])]

    for epoch_index, chunk in work.groupby("epoch_index", sort=True):
        label = majority_label(chunk[LABEL_COLUMN])
        if label not in label_map:
            continue
        if len(chunk) < samples_per_epoch * min_coverage:
            continue

        chunk = chunk.head(samples_per_epoch)
        wearable_values = chunk[WEARABLE_COLUMNS].to_numpy(dtype=np.float32)
        if wearable_values.shape[0] < samples_per_epoch:
            pad = np.full((samples_per_epoch - wearable_values.shape[0], len(WEARABLE_COLUMNS)), np.nan, dtype=np.float32)
            wearable_values = np.vstack([wearable_values, pad])

        x_wearable.append(wearable_values)
        y.append(label_map[label])

        if numeric_psg_columns:
            psg_values = chunk[numeric_psg_columns].to_numpy(dtype=np.float32)
            if psg_values.shape[0] < samples_per_epoch:
                pad = np.full((samples_per_epoch - psg_values.shape[0], len(numeric_psg_columns)), np.nan, dtype=np.float32)
                psg_values = np.vstack([psg_values, pad])
            x_psg.append(psg_values)

        meta_rows.append(
            {
                "subject_id": chunk["subject_id"].iloc[0],
                "epoch_index": int(epoch_index),
                "timestamp_start_s": float(epoch_index * epoch_seconds),
                "timestamp_end_s": float((epoch_index + 1) * epoch_seconds),
                "label": label,
                "label_id": label_map[label],
                "n_samples": int(len(chunk)),
            }
        )

    X_wearable = np.stack(x_wearable) if x_wearable else np.empty((0, samples_per_epoch, len(WEARABLE_COLUMNS)), dtype=np.float32)
    y_array = np.asarray(y, dtype=np.int64)
    X_psg = np.stack(x_psg) if x_psg else None
    meta = pd.DataFrame(meta_rows)
    return X_wearable, y_array, X_psg, meta

Xw_example, y_example, Xpsg_example, meta_example = epochise_record(example_df, psg_columns)
print("X wearable:", Xw_example.shape)
print("y:", y_example.shape, np.unique(y_example, return_counts=True))
print("X PSG:", None if Xpsg_example is None else Xpsg_example.shape)
display(meta_example.head())

## 5. Procesar todos los participantes

Esta celda concatena todos los epochs validos y guarda un `.npz` para entrenamiento. Tambien guarda metadatos por epoch en CSV.

In [ ]:
def process_all_records(signal_files: list[Path], max_subjects: int | None = None):
    all_xw, all_y, all_xpsg, all_meta = [], [], [], []
    psg_columns_reference = None

    selected_files = signal_files[:max_subjects] if max_subjects else signal_files
    for i, path in enumerate(selected_files, start=1):
        try:
            subject_id, df, _ = load_record(path)
            df, psg_columns = split_wearable_psg_label(df)

            if psg_columns_reference is None:
                psg_columns_reference = [col for col in psg_columns if pd.api.types.is_numeric_dtype(df[col])]
            else:
                psg_columns = [col for col in psg_columns_reference if col in df.columns]

            Xw, y, Xpsg, meta = epochise_record(df, psg_columns)
            if len(y) == 0:
                warnings.warn(f"Sin epochs validos para {subject_id}")
                continue

            all_xw.append(Xw)
            all_y.append(y)
            if Xpsg is not None:
                all_xpsg.append(Xpsg)
            all_meta.append(meta.assign(source_file=str(path.relative_to(DREAMT_ROOT))))
            print(f"[{i}/{len(selected_files)}] {subject_id}: {len(y)} epochs")
        except Exception as exc:
            warnings.warn(f"Error procesando {path}: {exc}")

    if not all_y:
        raise RuntimeError("No se generaron epochs. Revisa rutas, columnas y SAMPLE_RATE_HZ.")

    X_wearable = np.concatenate(all_xw, axis=0)
    y = np.concatenate(all_y, axis=0)
    X_psg = np.concatenate(all_xpsg, axis=0) if all_xpsg else None
    meta = pd.concat(all_meta, ignore_index=True)
    return X_wearable, y, X_psg, meta, psg_columns_reference or []

# Para una prueba rapida puedes usar max_subjects=3. Para el dataset completo, usa max_subjects=None.
X_wearable, y, X_psg, epoch_meta, psg_columns_used = process_all_records(signal_files, max_subjects=None)

print("Total X_wearable:", X_wearable.shape)
print("Total y:", y.shape)
print("Total X_psg:", None if X_psg is None else X_psg.shape)
display(epoch_meta["label"].value_counts().rename("epochs"))

In [ ]:
npz_path = OUTPUT_DIR / "dreamt_wearable_psg_epochs_30s.npz"
meta_path = OUTPUT_DIR / "dreamt_epoch_metadata.csv"
config_path = OUTPUT_DIR / "dreamt_extraction_config.json"

payload = {
    "X_wearable": X_wearable,
    "y": y,
    "wearable_columns": np.array(WEARABLE_COLUMNS),
    "label_names_5_class": np.array([label for label, _ in sorted(LABEL_MAP_5_CLASS.items(), key=lambda item: item[1])]),
}
if X_psg is not None:
    payload["X_psg"] = X_psg
    payload["psg_columns"] = np.array(psg_columns_used)

np.savez_compressed(npz_path, **payload)
epoch_meta.to_csv(meta_path, index=False)

config = {
    "dreamt_root": str(DREAMT_ROOT),
    "data_subdir": DATA_SUBDIR,
    "epoch_seconds": EPOCH_SECONDS,
    "sample_rate_hz": SAMPLE_RATE_HZ,
    "min_epoch_coverage": MIN_EPOCH_COVERAGE,
    "wearable_columns": WEARABLE_COLUMNS,
    "label_map_5_class": LABEL_MAP_5_CLASS,
    "psg_columns": psg_columns_used,
    "sources": [str(path.relative_to(DREAMT_ROOT)) for path in signal_files],
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Guardado:")
print("-", npz_path)
print("-", meta_path)
print("-", config_path)

## 6. Carga para entrenamiento

Ejemplo minimo para usar el archivo generado en un entrenamiento posterior.

In [ ]:
data = np.load(OUTPUT_DIR / "dreamt_wearable_psg_epochs_30s.npz", allow_pickle=True)
X_train_ready = data["X_wearable"]
y_train_ready = data["y"]

print("Entrada wearable:", X_train_ready.shape)
print("Ground truth PSG:", y_train_ready.shape)
print("Canales:", data["wearable_columns"])
print("Clases:", data["label_names_5_class"])

## Referencias

- DREAMT PhysioNet v2.2.0: Dataset for Real-time sleep stage EstimAtion using Multisensor wearable Technology.
- En `data_100Hz`, las senales wearable y PSG estan alineadas a 100 Hz. Las etiquetas `Sleep_Stage` proceden de anotacion tecnica de PSG cada 30 s.